In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
display(dbutils.fs.ls("file:/Workspace"))

path,name,size,modificationTime
file:/Workspace/.rm-rf-guard,.rm-rf-guard,0,1785906562171
file:/Workspace/Repos/,Repos/,4096,1785908744055
file:/Workspace/Users/,Users/,4096,1785908744055
file:/Workspace/Shared/,Shared/,4096,1785908744055


In [0]:
display(dbutils.fs.ls("file:/Workspace/Users"))

path,name,size,modificationTime
file:/Workspace/Users/Groups/,Groups/,4096,1785908791120
file:/Workspace/Users/harshanaydv@gmail.com/,harshanaydv@gmail.com/,4096,1785908791120


In [0]:
display(dbutils.fs.ls("file:/Workspace/Users/harshanaydv@gmail.com"))

path,name,size,modificationTime
file:/Workspace/Users/harshanaydv@gmail.com/.db_internal/,.db_internal/,4096,1785908804711
file:/Workspace/Users/harshanaydv@gmail.com/Drafts/,Drafts/,4096,1785908804711
file:/Workspace/Users/harshanaydv@gmail.com/CustomerSupportPipeline.ipynb,CustomerSupportPipeline.ipynb,2436,1785908804918
file:/Workspace/Users/harshanaydv@gmail.com/day2.csv,day2.csv,398,1785908314056
file:/Workspace/Users/harshanaydv@gmail.com/day1.csv,day1.csv,569,1785908314060
file:/Workspace/Users/harshanaydv@gmail.com/agents.csv,agents.csv,476,1785908314078


In [0]:
agents = spark.read.option("header", True).csv("file:/Workspace/Users/harshanaydv@gmail.com/agents.csv")

day1 = spark.read.option("header", True).csv("file:/Workspace/Users/harshanaydv@gmail.com/day1.csv")

day2 = spark.read.option("header", True).csv("file:/Workspace/Users/harshanaydv@gmail.com/day2.csv")

In [0]:
display(agents)
display(day1)
display(day2)

AgentID,AgentName,TeamLead,Department
A001,Rahul,TL01,Technical
A002,Priya,TL01,Technical
A003,Amit,TL02,Billing
A004,Neha,TL02,Billing
A005,Arjun,TL03,Technical
A006,Sneha,TL03,Technical
A007,Vikram,TL04,Sales
A008,Kavya,TL04,Sales
A009,Mohan,TL05,Support
A010,Riya,TL05,Support


TicketID,AgentID,Status,ResolutionTime
T001,A001,Resolved,0h 22m 45s
T002,A002,Pending,0h 12m 10s
T003,A003,Resolved,0h 35m 30s
T004,A004,Resolved,0h 14m 20s
T005,A005,Resolved,1h 05m 40s
T006,A006,Open,0h 05m 20s
T007,A007,Resolved,0h 28m 15s
T008,A008,Resolved,0h 16m 50s
T009,A009,Pending,0h 10m 10s
T010,A010,Resolved,0h 45m 15s


TicketID,AgentID,Status,ResolutionTime
T019,A002,Resolved,0h 18m 15s
T020,A004,Resolved,0h 25m 30s
T021,A006,Resolved,0h 40m 50s
T022,A009,Resolved,0h 30m 10s
T023,A011,Resolved,0h 19m 40s
T024,A015,Resolved,0h 17m 20s
T025,A017,Resolved,0h 23m 25s
T026,A018,Pending,0h 12m 30s
T027,A003,Resolved,0h 26m 15s
T028,A001,Resolved,0h 21m 40s


In [0]:
agents.printSchema()
day1.printSchema()
day2.printSchema()

root
 |-- AgentID: string (nullable = true)
 |-- AgentName: string (nullable = true)
 |-- TeamLead: string (nullable = true)
 |-- Department: string (nullable = true)

root
 |-- TicketID: string (nullable = true)
 |-- AgentID: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- ResolutionTime: string (nullable = true)

root
 |-- TicketID: string (nullable = true)
 |-- AgentID: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- ResolutionTime: string (nullable = true)



In [0]:
agents = agents.na.drop()

day1 = day1.na.drop()

day2 = day2.na.drop()

In [0]:
day1 = day1.filter(col("Status") == "Resolved")

day2 = day2.filter(col("Status") == "Resolved")

In [0]:
day1 = day1.join(agents, "AgentID")

day2 = day2.join(agents, "AgentID")

In [0]:
validTL = [
    "TL01",
    "TL02",
    "TL03",
    "TL04",
    "TL05",
    "TL06",
    "TL07",
    "TL08"
]

day1 = day1.filter(col("TeamLead").isin(validTL))

day2 = day2.filter(col("TeamLead").isin(validTL))

In [0]:
display(day1)

display(day2)

AgentID,TicketID,Status,ResolutionTime,AgentName,TeamLead,Department
A001,T001,Resolved,0h 22m 45s,Rahul,TL01,Technical
A003,T003,Resolved,0h 35m 30s,Amit,TL02,Billing
A004,T004,Resolved,0h 14m 20s,Neha,TL02,Billing
A005,T005,Resolved,1h 05m 40s,Arjun,TL03,Technical
A007,T007,Resolved,0h 28m 15s,Vikram,TL04,Sales
A008,T008,Resolved,0h 16m 50s,Kavya,TL04,Sales
A010,T010,Resolved,0h 45m 15s,Riya,TL05,Support
A011,T011,Resolved,0h 13m 50s,Ajay,TL06,Support
A012,T012,Resolved,0h 20m 20s,Meera,TL06,Technical
A013,T013,Resolved,0h 17m 30s,Karan,TL07,Billing


AgentID,TicketID,Status,ResolutionTime,AgentName,TeamLead,Department
A001,T028,Resolved,0h 21m 40s,Rahul,TL01,Technical
A002,T019,Resolved,0h 18m 15s,Priya,TL01,Technical
A003,T027,Resolved,0h 26m 15s,Amit,TL02,Billing
A004,T020,Resolved,0h 25m 30s,Neha,TL02,Billing
A005,T030,Resolved,0h 27m 55s,Arjun,TL03,Technical
A006,T021,Resolved,0h 40m 50s,Sneha,TL03,Technical
A009,T022,Resolved,0h 30m 10s,Mohan,TL05,Support
A010,T029,Resolved,0h 33m 10s,Riya,TL05,Support
A011,T023,Resolved,0h 19m 40s,Ajay,TL06,Support
A015,T024,Resolved,0h 17m 20s,Nikhil,TL08,Support


In [0]:
from pyspark.sql.functions import regexp_extract

day1 = day1.withColumn("Hours", regexp_extract("ResolutionTime", r"(\d+)h", 1).cast("int")) \
           .withColumn("Minutes", regexp_extract("ResolutionTime", r"(\d+)m", 1).cast("int")) \
           .withColumn("Seconds", regexp_extract("ResolutionTime", r"(\d+)s", 1).cast("int")) \
           .withColumn(
               "ResolutionMinutes",
               col("Hours") * 60 + col("Minutes") + col("Seconds") / 60
           )

day2 = day2.withColumn("Hours", regexp_extract("ResolutionTime", r"(\d+)h", 1).cast("int")) \
           .withColumn("Minutes", regexp_extract("ResolutionTime", r"(\d+)m", 1).cast("int")) \
           .withColumn("Seconds", regexp_extract("ResolutionTime", r"(\d+)s", 1).cast("int")) \
           .withColumn(
               "ResolutionMinutes",
               col("Hours") * 60 + col("Minutes") + col("Seconds") / 60
           )

In [0]:
display(day1)

display(day2)

AgentID,TicketID,Status,ResolutionTime,AgentName,TeamLead,Department,Hours,Minutes,Seconds,ResolutionMinutes
A001,T001,Resolved,0h 22m 45s,Rahul,TL01,Technical,0,22,45,22.75
A003,T003,Resolved,0h 35m 30s,Amit,TL02,Billing,0,35,30,35.5
A004,T004,Resolved,0h 14m 20s,Neha,TL02,Billing,0,14,20,14.333333333333334
A005,T005,Resolved,1h 05m 40s,Arjun,TL03,Technical,1,5,40,65.66666666666667
A007,T007,Resolved,0h 28m 15s,Vikram,TL04,Sales,0,28,15,28.25
A008,T008,Resolved,0h 16m 50s,Kavya,TL04,Sales,0,16,50,16.833333333333332
A010,T010,Resolved,0h 45m 15s,Riya,TL05,Support,0,45,15,45.25
A011,T011,Resolved,0h 13m 50s,Ajay,TL06,Support,0,13,50,13.833333333333334
A012,T012,Resolved,0h 20m 20s,Meera,TL06,Technical,0,20,20,20.333333333333332
A013,T013,Resolved,0h 17m 30s,Karan,TL07,Billing,0,17,30,17.5


AgentID,TicketID,Status,ResolutionTime,AgentName,TeamLead,Department,Hours,Minutes,Seconds,ResolutionMinutes
A001,T028,Resolved,0h 21m 40s,Rahul,TL01,Technical,0,21,40,21.666666666666668
A002,T019,Resolved,0h 18m 15s,Priya,TL01,Technical,0,18,15,18.25
A003,T027,Resolved,0h 26m 15s,Amit,TL02,Billing,0,26,15,26.25
A004,T020,Resolved,0h 25m 30s,Neha,TL02,Billing,0,25,30,25.5
A005,T030,Resolved,0h 27m 55s,Arjun,TL03,Technical,0,27,55,27.916666666666668
A006,T021,Resolved,0h 40m 50s,Sneha,TL03,Technical,0,40,50,40.833333333333336
A009,T022,Resolved,0h 30m 10s,Mohan,TL05,Support,0,30,10,30.166666666666668
A010,T029,Resolved,0h 33m 10s,Riya,TL05,Support,0,33,10,33.166666666666664
A011,T023,Resolved,0h 19m 40s,Ajay,TL06,Support,0,19,40,19.666666666666668
A015,T024,Resolved,0h 17m 20s,Nikhil,TL08,Support,0,17,20,17.333333333333332


In [0]:
day1 = day1.filter(col("ResolutionMinutes") >= 15)

day2 = day2.filter(col("ResolutionMinutes") >= 15)

In [0]:
display(day1)

display(day2)

AgentID,TicketID,Status,ResolutionTime,AgentName,TeamLead,Department,Hours,Minutes,Seconds,ResolutionMinutes
A001,T001,Resolved,0h 22m 45s,Rahul,TL01,Technical,0,22,45,22.75
A003,T003,Resolved,0h 35m 30s,Amit,TL02,Billing,0,35,30,35.5
A005,T005,Resolved,1h 05m 40s,Arjun,TL03,Technical,1,5,40,65.66666666666667
A007,T007,Resolved,0h 28m 15s,Vikram,TL04,Sales,0,28,15,28.25
A008,T008,Resolved,0h 16m 50s,Kavya,TL04,Sales,0,16,50,16.833333333333332
A010,T010,Resolved,0h 45m 15s,Riya,TL05,Support,0,45,15,45.25
A012,T012,Resolved,0h 20m 20s,Meera,TL06,Technical,0,20,20,20.333333333333332
A013,T013,Resolved,0h 17m 30s,Karan,TL07,Billing,0,17,30,17.5
A014,T014,Resolved,0h 30m 05s,Pooja,TL07,Billing,0,30,5,30.083333333333332
A016,T016,Resolved,0h 18m 25s,Anjali,TL08,Technical,0,18,25,18.416666666666668


AgentID,TicketID,Status,ResolutionTime,AgentName,TeamLead,Department,Hours,Minutes,Seconds,ResolutionMinutes
A001,T028,Resolved,0h 21m 40s,Rahul,TL01,Technical,0,21,40,21.666666666666668
A002,T019,Resolved,0h 18m 15s,Priya,TL01,Technical,0,18,15,18.25
A003,T027,Resolved,0h 26m 15s,Amit,TL02,Billing,0,26,15,26.25
A004,T020,Resolved,0h 25m 30s,Neha,TL02,Billing,0,25,30,25.5
A005,T030,Resolved,0h 27m 55s,Arjun,TL03,Technical,0,27,55,27.916666666666668
A006,T021,Resolved,0h 40m 50s,Sneha,TL03,Technical,0,40,50,40.833333333333336
A009,T022,Resolved,0h 30m 10s,Mohan,TL05,Support,0,30,10,30.166666666666668
A010,T029,Resolved,0h 33m 10s,Riya,TL05,Support,0,33,10,33.166666666666664
A011,T023,Resolved,0h 19m 40s,Ajay,TL06,Support,0,19,40,19.666666666666668
A015,T024,Resolved,0h 17m 20s,Nikhil,TL08,Support,0,17,20,17.333333333333332


In [0]:
resolved_agents = day1.select("AgentID").distinct()

day2 = day2.join(
    resolved_agents,
    on="AgentID",
    how="left_anti"
)

In [0]:
display(day2)

AgentID,TicketID,Status,ResolutionTime,AgentName,TeamLead,Department,Hours,Minutes,Seconds,ResolutionMinutes
A002,T019,Resolved,0h 18m 15s,Priya,TL01,Technical,0,18,15,18.25
A004,T020,Resolved,0h 25m 30s,Neha,TL02,Billing,0,25,30,25.5
A006,T021,Resolved,0h 40m 50s,Sneha,TL03,Technical,0,40,50,40.833333333333336
A009,T022,Resolved,0h 30m 10s,Mohan,TL05,Support,0,30,10,30.166666666666668
A011,T023,Resolved,0h 19m 40s,Ajay,TL06,Support,0,19,40,19.666666666666668
A015,T024,Resolved,0h 17m 20s,Nikhil,TL08,Support,0,17,20,17.333333333333332


In [0]:
team_performance = day1.groupBy("TeamLead").agg(
    count("TicketID").alias("ResolvedTickets"),
    round(avg("ResolutionMinutes"), 2).alias("AverageResolutionTime")
)

display(team_performance)

TeamLead,ResolvedTickets,AverageResolutionTime
TL06,1,20.33
TL02,1,35.5
TL08,1,18.42
TL04,2,22.54
TL03,1,65.67
TL07,2,23.79
TL01,1,22.75
TL05,1,45.25


In [0]:
agent_performance = day1.groupBy(
    "AgentID",
    "AgentName"
).agg(
    count("TicketID").alias("ResolvedTickets"),
    round(avg("ResolutionMinutes"), 2).alias("AverageResolutionTime")
)

display(agent_performance)

AgentID,AgentName,ResolvedTickets,AverageResolutionTime
A016,Anjali,1,18.42
A005,Arjun,1,65.67
A001,Rahul,1,22.75
A003,Amit,1,35.5
A010,Riya,1,45.25
A013,Karan,1,17.5
A014,Pooja,1,30.08
A007,Vikram,1,28.25
A008,Kavya,1,16.83
A012,Meera,1,20.33


In [0]:
day1.write.mode("overwrite").parquet(
    "file:/Workspace/Users/harshanaydv@gmail.com/processed_day1"
)

day2.write.mode("overwrite").parquet(
    "file:/Workspace/Users/harshanaydv@gmail.com/processed_day2"
)

team_performance.write.mode("overwrite").parquet(
    "file:/Workspace/Users/harshanaydv@gmail.com/team_performance"
)

agent_performance.write.mode("overwrite").parquet(
    "file:/Workspace/Users/harshanaydv@gmail.com/agent_performance"
)

In [0]:
print("Agents Count:", agents.count())
print("Day1 Tickets:", day1.count())
print("Day2 Tickets:", day2.count())

Agents Count: 18
Day1 Tickets: 10
Day2 Tickets: 6


In [0]:
display(
    team_performance.orderBy(col("ResolvedTickets").desc())
)

TeamLead,ResolvedTickets,AverageResolutionTime
TL04,2,22.54
TL07,2,23.79
TL06,1,20.33
TL02,1,35.5
TL08,1,18.42
TL03,1,65.67
TL01,1,22.75
TL05,1,45.25


In [0]:
display(
    agent_performance.orderBy(col("ResolvedTickets").desc())
)

AgentID,AgentName,ResolvedTickets,AverageResolutionTime
A016,Anjali,1,18.42
A005,Arjun,1,65.67
A001,Rahul,1,22.75
A003,Amit,1,35.5
A010,Riya,1,45.25
A013,Karan,1,17.5
A014,Pooja,1,30.08
A007,Vikram,1,28.25
A008,Kavya,1,16.83
A012,Meera,1,20.33


In [0]:
from pyspark.sql.functions import avg, max, min

day1.select(
    avg("ResolutionMinutes").alias("Average Resolution Time"),
    max("ResolutionMinutes").alias("Maximum Resolution Time"),
    min("ResolutionMinutes").alias("Minimum Resolution Time")
).show()

+-----------------------+-----------------------+-----------------------+
|Average Resolution Time|Maximum Resolution Time|Minimum Resolution Time|
+-----------------------+-----------------------+-----------------------+
|     30.058333333333337|      65.66666666666667|     16.833333333333332|
+-----------------------+-----------------------+-----------------------+

